In [ ]:
# ==========================================
# СЕКЦІЯ 1: Install deps & Setup
# ==========================================
!pip install -q groq jsonschema pandas
import os
import sys
import json
from groq import Groq

# Клонування репозиторію
os.chdir('/content')
!rm -rf NLP_Course
!git clone https://github.com/dmytroslav/NLP_Course.git
sys.path.append('/content/NLP_Course')

# Налаштування клієнта Groq
API_KEY = "YOUR_API_KEY_HERE"
client = Groq(api_key=API_KEY)

# Функція-обгортка для LLM (передається у Flow)
def llm_client(prompt: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.choices[0].message.content

# Очищення старих логів перед новим запуском
log_path = '/content/NLP_Course/docs/flow_logs_lab14.jsonl'
if os.path.exists(log_path):
    os.remove(log_path)

# Імпорт класів для Лабораторної 14
from src.flow import NewsAnalysisFlow
from src.flow_logger import FlowLogger

print("Setup completed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.6 MB/s eta 0:00:00
Cloning into 'NLP_Course'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 317 (delta 9), reused 27 (delta 7), pack-reused 283 (from 1)
Receiving objects: 100% (317/317), 2.17 MiB | 13.24 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Setup completed successfully!


In [2]:
test_cases = [
    {"id": "case_01", "text": "Офіційно: Київ залишається столицею України."},
    {"id": "case_02", "text": "ТЕРМІНОВО! Завтра всім вимкнуть світло назавжди! Репост!"},
    {"id": "case_03", "text": "Продам гараж у центрі міста, недорого, писати в ПП."},
    {"id": "case_04", "text": "З 1 січня 2026 року податок на додану вартість зросте до 25%."},
    {"id": "case_05", "text": "Канал 'Блискавка' повідомляє, що вчені довели пласкість Землі."},
    {"id": "case_06", "text": "Привіт, як справи? Підеш сьогодні на каву після пар?"},
    {"id": "case_07", "text": "ШОК! Знайдено ліки від усіх хвороб, достатньо простої соди."},
    {"id": "case_08", "text": "Курс долара НБУ станом на сьогодні становить 41 гривню."},
    {"id": "case_09", "text": "РИА Новости: Усіх пенсіонерів позбавлять виплат з наступного місяця."},
    {"id": "case_10", "text": "Міністерство освіти затвердило нові правила вступу на магістратуру."}
]

In [3]:
from src.flow import NewsAnalysisFlow
from src.flow_logger import FlowLogger

flow = NewsAnalysisFlow(llm_client)
logger = FlowLogger("docs/flow_logs_lab14.jsonl")
results = []

print("Початок обробки тестових кейсів...\n")

for case in test_cases:
    print(f"Обробка {case['id']}...")
    result = flow.run(case["id"], case["text"])
    logger.log_case(result)
    results.append(result)

print("\nОбробку завершено. Логи збережено у docs/flow_logs_lab14.jsonl")

Початок обробки тестових кейсів...

Обробка case_01...
Обробка case_02...
Обробка case_03...
Обробка case_04...
Обробка case_05...
Обробка case_06...
Обробка case_07...
Обробка case_08...
Обробка case_09...
Обробка case_10...

Обробку завершено. Логи збережено у docs/flow_logs_lab14.jsonl


In [4]:
total_cases = len(results)
routed_cases = sum(1 for r in results if r["route"] == "analyze_news")

flow_completed = sum(1 for r in results if r["final_status"] in ["completed_success", "completed_skipped", "completed_with_errors"])
validation_passed_initially = sum(1 for r in results if r.get("validation_result", {}).get("is_valid") and not r.get("fallback_triggered"))
fallbacks_triggered = sum(1 for r in results if r.get("fallback_triggered"))
fallbacks_successful = sum(1 for r in results if r.get("fallback_triggered") and r.get("validation_result", {}).get("is_valid"))
export_valid = sum(1 for r in results if r["export_output"]["status"] == "success")

flow_completion_rate = (flow_completed / total_cases) * 100
validation_pass_rate = (validation_passed_initially / routed_cases * 100) if routed_cases else 0
fallback_activation_rate = (fallbacks_triggered / routed_cases * 100) if routed_cases else 0
fallback_success_rate = (fallbacks_successful / fallbacks_triggered * 100) if fallbacks_triggered else 0
export_valid_rate = (export_valid / routed_cases * 100) if routed_cases else 0

print(f"=== МЕТРИКИ FLOW ORCHESTRATION ===")
print(f"Flow Completion Rate:        {flow_completion_rate:.1f}%")
print(f"Validation Pass Rate (Init): {validation_pass_rate:.1f}%")
print(f"Fallback Activation Rate:    {fallback_activation_rate:.1f}%")
print(f"Fallback Success Rate:       {fallback_success_rate:.1f}%")
print(f"Export Valid Rate:           {export_valid_rate:.1f}%")

=== МЕТРИКИ FLOW ORCHESTRATION ===
Flow Completion Rate:        100.0%
Validation Pass Rate (Init): 100.0%
Fallback Activation Rate:    0.0%
Fallback Success Rate:       0.0%
Export Valid Rate:           100.0%


## 4. Error Analysis (Детальний розбір 10 кейсів)

**Case 01:** "Офіційно: Київ залишається столицею України."
* **Expected behavior:** Route `analyze_news`, verdict `True`.
* **Actual route / Execute:** `analyze_news`, витягнуто факт "Київ - столиця".
* **Final status:** `completed_success`.

**Case 02:** "ТЕРМІНОВО! Завтра всім вимкнуть світло назавжди! Репост!"
* **Expected behavior:** Route `analyze_news`, verdict `Fake`.
* **Actual route / Execute:** `analyze_news`, verdict `Fake`, sources: [].
* **Final status:** `completed_success`.

**Case 03:** "Продам гараж у центрі міста, недорого, писати в ПП."
* **Expected behavior:** Route `skip` (це оголошення, не новина).
* **Actual route / Execute:** Роутер помилився -> `analyze_news`. Екзекутор відпрацював як `Unknown`.
* **Error category:** `wrong route`.
* **Possible fix:** Додати приклади комерційних оголошень у prompt роутера.

**Case 04:** "З 1 січня 2026 року податок на додану вартість зросте до 25%."
* **Expected behavior:** Route `analyze_news`, verdict `Unknown` або `Fake` (залежить від знань моделі).
* **Actual route / Execute:** `analyze_news`, verdict `True` (модель прийняла твердження за факт).
* **Error category:** `hallucination / over-trust`.
* **Possible fix:** Підключити RAG (інструмент пошуку) на етапі Execute.

**Case 05:** "Канал 'Блискавка' повідомляє, що вчені довели пласкість Землі."
* **Expected behavior:** Route `analyze_news`, verdict `Fake`.
* **Actual route / Execute:** `analyze_news`, verdict `Unknown` (модель завагалася щодо наукових даних).
* **Error category:** `low confidence accuracy`.

**Case 06:** "Привіт, як справи? Підеш сьогодні на каву після пар?"
* **Expected behavior:** Route `skip`.
* **Actual route / Execute:** `skip`. Executor не викликався.
* **Final status:** `completed_skipped`. Ідеальна робота роутера.

**Case 07:** "ШОК! Знайдено ліки від усіх хвороб, достатньо простої соди."
* **Expected behavior:** Route `analyze_news`, verdict `Fake`.
* **Actual route / Execute:** `analyze_news`, verdict `Fake`.
* **Final status:** `completed_success`.

**Case 08:** "Курс долара НБУ станом на сьогодні становить 41 гривню."
* **Expected behavior:** Route `analyze_news`.
* **Actual route / Execute:** `analyze_news`, verdict `Unknown` (відсутність live-даних).
* **Final status:** `completed_success`. Безпечний збій (safe failure).

**Case 09:** "РИА Новости: Усіх пенсіонерів позбавлять виплат з наступного місяця."
* **Expected behavior:** Route `analyze_news`, verdict `Fake/Unknown`.
* **Actual route / Execute:** `analyze_news`, verdict `Unknown`.
* **Final status:** `completed_success`.

**Case 10:** "Міністерство освіти затвердило нові правила вступу на магістратуру."
* **Expected behavior:** Route `analyze_news`.
* **Actual route / Execute:** `analyze_news`, verdict `Unknown`.
* **Final status:** `completed_success`.

## 5. Порівняння з ad-hoc pipeline
Без `Stateful Flow` (у варіанті `input -> model -> output`) ми мали проблему: якщо модель віддавала неповний JSON, код падав із помилкою `KeyError`. Впровадження Flow зробило процес детермінованим: тепер помилка відловлюється у `Validator`, і якщо `Fallback` не справляється, ми все одно отримуємо стабільний JSON через `Exporter` (із порожніми значеннями та статусом "failed"), що гарантує відсутність крашів у продакшені.